# RetinaMNIST Representations Experiment (PCA vs AE vs MAE)

This Colab notebook orchestrates a complete experiment on RetinaMNIST:
- **PCA** on flattened pixels
- **Convolutional Autoencoder (AE)** latent vectors
- **Pretrained ViT/MAE encoder** CLS-token features (via `timm`)

All three representations are evaluated with the **same MLP classifier head** (trained separately per representation).

All outputs (plots + saved figures) are written to `./outputs/`.

## SECTION 0 — Setup

In [ ]:
# Install dependencies (Colab)
!pip -q install medmnist timm umap-learn ipywidgets seaborn tqdm scikit-learn

import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# (Option A) Clone project repo (uncomment and set your repo)
# !git clone https://github.com/<USERNAME>/<REPO>.git
# %cd <REPO>

# (Option B) Mount Google Drive (uncomment)
# from google.colab import drive
# drive.mount('/content/drive')

# Add project root to sys.path
PROJECT_ROOT = Path.cwd()
if not PROJECT_ROOT.exists():
    raise FileNotFoundError("Project root not found; cd into the repo directory first.")
sys.path.insert(0, str(PROJECT_ROOT))

# Import modules
from configs.config import Config, set_global_seed
from data.dataset import load_medmnist
from representations.pca_repr import fit_transform_pca
from representations.ae_repr import train_and_extract_ae_representations
from representations.mae_repr import extract_mae_representations_multi
from models.classifier import train_mlp
from utils.timer import Timer, log_time
from utils.io import save_json
from utils.metrics import compute_metrics, print_metrics_table
from utils.visualization import (
    plot_training_curves,
    plot_epoch_times,
    plot_reconstruction_samples,
    plot_tsne,
    plot_final_comparison,
)

os.makedirs(PROJECT_ROOT / "outputs", exist_ok=True)
os.makedirs(PROJECT_ROOT / "results", exist_ok=True)
print("Project root:", PROJECT_ROOT)

## SECTION 1 — Config & Dry Run Toggle

In [ ]:
# DRY RUN toggle
DRY_RUN = True   # ← CHANGE THIS TO FALSE FOR FULL TRAINING

# Datasets to run (loop over all)
DATASET_NAMES = [
    "dermamnist",
    "pathmnist",
    "retinamnist",
    "pneumoniamnist",
]

# Base config (dataset_name will be overridden inside the loop)
config = Config(DRY_RUN=DRY_RUN, DATASET_SIZE=224)
set_global_seed(config.SEED)
print("Base config:", config)

if config.DRY_RUN:
    print("🔍 DRY RUN MODE — using subsampled data")
else:
    print("🚀 FULL RUN MODE")

print("Datasets:", DATASET_NAMES)

## SECTION 2 — Data Loading

In [ ]:
from dataclasses import replace

def class_dist(y: np.ndarray) -> dict:
    y = np.asarray(y).reshape(-1)
    vals, counts = np.unique(y, return_counts=True)
    return dict(zip(vals.tolist(), counts.tolist()))

ALL_RESULTS = {}

for ds_name in DATASET_NAMES:
    print("\n" + "=" * 80)
    print(f"DATASET: {ds_name}")
    print("=" * 80)

    cfg = replace(config, DATASET_NAME=ds_name)
    set_global_seed(cfg.SEED)

    # SECTION 2 — Data Loading
    bundle = load_medmnist(cfg, dataset_name=ds_name, dry_run=cfg.DRY_RUN, num_workers=2)
    (x_train, y_train) = bundle.arrays["train"]
    (x_val, y_val) = bundle.arrays["val"]
    (x_test, y_test) = bundle.arrays["test"]

    print("n_train:", len(x_train), "n_val:", len(x_val), "n_test:", len(x_test))
    print("Class dist (train):", class_dist(y_train))

    # Save sample grid (handle grayscale vs RGB)
    out_dir = PROJECT_ROOT / "outputs" / ds_name
    os.makedirs(out_dir, exist_ok=True)
    fig, axes = plt.subplots(4, 4, figsize=(6, 6))
    idxs = np.random.choice(len(x_train), size=min(16, len(x_train)), replace=False)
    for ax, idx in zip(axes.flatten(), idxs):
        img = x_train[idx]
        if img.shape[0] == 1:
            ax.imshow(img[0], cmap="gray", vmin=0, vmax=1)
        else:
            ax.imshow(np.transpose(img, (1, 2, 0)))
        ax.set_title(f"y={int(y_train[idx])}")
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_dir / "samples_grid.png", dpi=200, bbox_inches="tight")
    plt.show()

    # SECTION 3 — PCA
    pca_features, pca_model, pca_fit_time, pca_transform_times = fit_transform_pca(x_train, x_val, x_test, cfg)
    pca_feature_time = float(pca_fit_time + sum(pca_transform_times.values()))
    log_time(f"{ds_name} PCA fit", pca_fit_time)

    # SECTION 4 — AE
    ae_train_loader = bundle.loaders["train"]
    ae_val_loader = bundle.loaders["val"]
    ae_start = time.time()
    ae_features, ae_history, ae_model = train_and_extract_ae_representations(
        ae_train_loader, ae_val_loader, x_train, x_val, x_test, cfg
    )
    ae_feature_time = float(time.time() - ae_start)

    # AE plots
    plt.figure(figsize=(6, 4))
    plt.plot(ae_history["epoch_losses"], label="Train recon loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(f"AE Training Loss — {ds_name}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "ae_loss_curve.png", dpi=200, bbox_inches="tight")
    plt.show()

    plot_epoch_times(
        ae_history["epoch_times"],
        title=f"AE Epoch Times — {ds_name}",
        save_path=str(out_dir / "ae_epoch_times.png"),
    )
    plt.show()

    plot_reconstruction_samples(
        ae_model,
        ae_train_loader,
        device=cfg.DEVICE,
        n=8,
        save_path=str(out_dir / "ae_recon_samples.png"),
    )
    plt.show()

    # SECTION 5 — MAE multi
    mae_features_by_model, mae_times_by_model = extract_mae_representations_multi(
        x_train, x_val, x_test, cfg, model_names=cfg.MAE_MODEL_NAMES, batch_size=256
    )

    # SECTION 6 — Train MLPs
    def train_one_method(method_name: str, feats: dict, y_tr, y_va, cfg_: Config, num_classes: int):
        print(f"\n=== Training MLP for {method_name} ({ds_name}) ===")
        start = time.time()
        model, hist = train_mlp(
            feats["train"],
            y_tr,
            feats["val"],
            y_va,
            cfg_,
            input_dim=int(feats["train"].shape[1]),
            num_classes=num_classes,
        )
        train_time = time.time() - start
        log_time(f"{method_name} MLP train", train_time)
        return model, hist, train_time

    num_classes = int(len(np.unique(y_train)))
    feature_sets = {"PCA": pca_features, "AE": ae_features}
    feature_times = {"PCA": pca_feature_time, "AE": ae_feature_time}
    for name in cfg.MAE_MODEL_NAMES:
        key = f"MAE:{name}"
        feature_sets[key] = mae_features_by_model[name]
        feature_times[key] = float(sum(mae_times_by_model[name].values()))

    mlp_models = {}
    mlp_histories = {}
    mlp_train_times = {}
    for method_name, feats in feature_sets.items():
        mlp_models[method_name], mlp_histories[method_name], mlp_train_times[method_name] = train_one_method(
            method_name, feats, y_train, y_val, cfg, num_classes
        )

    # SECTION 7 — Evaluation
    @torch.no_grad()
    def predict_numpy(model: torch.nn.Module, x: np.ndarray, batch_size: int = 256) -> np.ndarray:
        model.eval()
        preds = []
        for i in range(0, len(x), batch_size):
            xb = torch.from_numpy(np.asarray(x[i:i+batch_size], dtype=np.float32)).to(cfg.DEVICE)
            logits = model(xb)
            yhat = torch.argmax(logits, dim=1).detach().cpu().numpy().astype(int)
            preds.append(yhat)
        return np.concatenate(preds, axis=0)

    results = {}
    for method_name in mlp_models.keys():
        x_te = feature_sets[method_name]["test"]
        y_pred = predict_numpy(mlp_models[method_name], x_te)
        m = compute_metrics(y_test, y_pred)
        m["train_time"] = float(mlp_train_times[method_name])
        m["feature_time"] = float(feature_times[method_name])
        results[method_name] = m

    print_metrics_table(results)

    # Save JSON
    payload = {
        "dataset": ds_name,
        "config": {
            "DRY_RUN": cfg.DRY_RUN,
            "SEED": cfg.SEED,
            "BATCH_SIZE": cfg.BATCH_SIZE,
            "AE_EPOCHS": cfg.AE_EPOCHS,
            "AE_LR": cfg.AE_LR,
            "AE_LATENT_DIM": cfg.AE_LATENT_DIM,
            "PCA_N_COMPONENTS": cfg.PCA_N_COMPONENTS,
            "MLP_HIDDEN_DIMS": cfg.MLP_HIDDEN_DIMS,
            "MLP_EPOCHS": cfg.MLP_EPOCHS,
            "MLP_LR": cfg.MLP_LR,
            "MAE_MODEL_NAMES": cfg.MAE_MODEL_NAMES,
            "DATASET_SIZE": cfg.DATASET_SIZE,
        },
        "results": results,
    }
    save_json(str(PROJECT_ROOT / cfg.RESULTS_DIR / f"results_{ds_name}.json"), payload)
    ALL_RESULTS[ds_name] = results

print("\nDone. Wrote per-dataset JSON files to:", PROJECT_ROOT / config.RESULTS_DIR)

## SECTION 3 — PCA Representations

In [ ]:
# Handled inside the dataset loop cell in SECTION 2.
# This cell is kept for backward compatibility with older single-dataset runs.

## SECTION 4 — Autoencoder Training

In [ ]:
# Handled inside the dataset loop cell in SECTION 2.
# This cell is kept for backward compatibility with older single-dataset runs.

## SECTION 5 — MAE Feature Extraction

In [ ]:
# Handled inside the dataset loop cell in SECTION 2.
# This cell is kept for backward compatibility with older single-dataset runs.

## SECTION 6 — MLP Classifier Training (All Three)

In [ ]:
# Handled inside the dataset loop cell in SECTION 2.
# This cell is kept for backward compatibility with older single-dataset runs.

## SECTION 7 — Evaluation on Test Set

In [ ]:
# Handled inside the dataset loop cell in SECTION 2.
# This cell is kept for backward compatibility with older single-dataset runs.

## SECTION 8 — Visualizations

In [ ]:
# Handled inside the dataset loop cell in SECTION 2.
# This cell is kept for backward compatibility with older single-dataset runs.

## SECTION 9 — Summary Cell

In [ ]:
# Handled inside the dataset loop cell in SECTION 2.
# This cell is kept for backward compatibility with older single-dataset runs.